In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import pickle
import warnings, os
 
from sklearn.preprocessing import OrdinalEncoder
from sklearn.linear_model import LogisticRegression
 
warnings.filterwarnings("ignore")
os.makedirs("outputs", exist_ok=True)
os.makedirs("data", exist_ok=True)

### LOAD & PREPARE

In [2]:
df = pd.read_parquet("D:/project/Global AI Adoption & Workforce Impact/Notebook/data/cleaned.parquet")

In [3]:
CATEGORICAL = [
    "region", "industry", "company_size",
    "ai_primary_tool", "ai_use_case", "data_privacy_level",
]
enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
df[CATEGORICAL] = enc.fit_transform(df[CATEGORICAL])
for col in df.select_dtypes(include=["object", "string"]).columns:
    df[col] = OrdinalEncoder(
        handle_unknown="use_encoded_value", unknown_value=-1
    ).fit_transform(df[[col]])
df = df.apply(lambda c: c.cat.codes if hasattr(c, "cat") else c)

In [4]:
TREATMENT = "has_ethics_committee"
OUTCOMES  = ["revenue_growth_percent", "ai_failure_rate", "ai_maturity_score"]
CONTROLS  = [
    "log_annual_revenue_usd_millions", "log_num_employees", "company_age",
    "region", "industry", "company_size", "years_using_ai", "ai_budget_percentage",
    "regulatory_compliance_score", "ai_risk_management_score", "data_privacy_ord",
    "adoption_stage_ord", "num_ai_tools_used", "ai_training_hours",
]

In [5]:
# Treatment balance diagnostic
t_rate = df[TREATMENT].mean()
print(f"\n  Treatment prevalence: {t_rate:.4f} ({df[TREATMENT].sum():,.0f} / {len(df):,} rows)")
print("  NOTE: Near-perfect treatment — propensity matching not viable.")
print("  Using OLS regression adjustment (controls for observed confounders).\n")


  Treatment prevalence: 0.4101 (61,521 / 150,000 rows)
  NOTE: Near-perfect treatment — propensity matching not viable.
  Using OLS regression adjustment (controls for observed confounders).



### OLS REGRESSION ADJUSTMENT

In [6]:
print("=== Analysis 1: OLS regression adjustment ===")
ols_results = {}
for outcome in OUTCOMES:
    cols = [outcome, TREATMENT] + CONTROLS
    sub  = df[cols].dropna().astype(float)
    X    = sm.add_constant(sub[[TREATMENT] + CONTROLS])
    m    = sm.OLS(sub[outcome], X).fit()
 
    coef = m.params[TREATMENT]
    ci   = m.conf_int().loc[TREATMENT]
    pval = m.pvalues[TREATMENT]
    ols_results[outcome] = {
        "coef": coef, "ci_lo": ci[0], "ci_hi": ci[1], "pval": pval
    }
    sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else "ns"
    print(f"  {outcome:<35} coef={coef:+.4f}  "
          f"95%CI [{ci[0]:+.4f},{ci[1]:+.4f}]  p={pval:.4f} {sig}")

=== Analysis 1: OLS regression adjustment ===
  revenue_growth_percent              coef=-0.0370  95%CI [-0.0956,+0.0216]  p=0.2158 ns
  ai_failure_rate                     coef=+0.0932  95%CI [+0.0334,+0.1529]  p=0.0022 **
  ai_maturity_score                   coef=-0.0029  95%CI [-0.0034,-0.0023]  p=0.0000 ***


### FAILURE RISK DRIVERS (LOGISTIC)

In [7]:
print("\n=== Analysis 2: Failure risk drivers (logistic odds ratios) ===")
df["high_failure"] = (df["ai_failure_rate"] > df["ai_failure_rate"].median()).astype(int)
 
RISK_FEATURES = [
    "has_ethics_committee", "ai_risk_management_score", "regulatory_compliance_score",
    "data_privacy_ord", "adoption_stage_ord", "ai_budget_percentage",
    "ai_training_hours", "years_using_ai", "num_ai_tools_used",
    "log_annual_revenue_usd_millions", "company_size",
]
X_risk = df[RISK_FEATURES].dropna().astype(float)
y_risk = df.loc[X_risk.index, "high_failure"]
 
lr = LogisticRegression(max_iter=300, random_state=42, n_jobs=-1)
lr.fit(X_risk, y_risk)
risk_fi = pd.Series(np.exp(lr.coef_[0]), index=RISK_FEATURES).sort_values(ascending=False)
print(risk_fi.round(4).to_string())


=== Analysis 2: Failure risk drivers (logistic odds ratios) ===
company_size                       1.0326
years_using_ai                     1.0114
has_ethics_committee               1.0068
data_privacy_ord                   1.0048
regulatory_compliance_score        0.9999
ai_risk_management_score           0.9970
num_ai_tools_used                  0.9950
log_annual_revenue_usd_millions    0.9946
ai_budget_percentage               0.9574
ai_training_hours                  0.8903
adoption_stage_ord                 0.2747


### SUBGROUP: REVENUE BY ETHICS × MATURITY TIER

In [8]:
print("\n=== Analysis 3: Revenue by ethics × maturity tier ===")
df["maturity_tier"] = pd.qcut(df["ai_maturity_score"], q=3, labels=["Low", "Mid", "High"])
pivot = (
    df.groupby(["has_ethics_committee", "maturity_tier"], observed=True)
    ["revenue_growth_percent"].mean().unstack()
)
print(pivot.round(3))


=== Analysis 3: Revenue by ethics × maturity tier ===
maturity_tier           Low    Mid   High
has_ethics_committee                     
0                     2.274  4.467  6.911
1                     2.337  4.578  7.166


### INTERACTION DATA FOR PLOT

In [9]:
interaction_data = {}
for has_eth in [0, 1]:
    sub = df[df[TREATMENT] == has_eth].copy()
    sub["mat_bin"] = pd.qcut(sub["ai_maturity_score"], q=10, duplicates="drop")
    grp = sub.groupby("mat_bin", observed=True)["revenue_growth_percent"].mean()
    x_vals = [iv.mid for iv in grp.index.categories if iv in grp.index]
    interaction_data[has_eth] = (x_vals, grp.values.tolist())

#### Plots

In [11]:
sns.set_theme(style="whitegrid")
fig = plt.figure(figsize=(16, 11))

<Figure size 1600x1100 with 0 Axes>

In [ ]:
# Plot 1: OLS treatment coefficients with CI + significance
ax1 = fig.add_subplot(2, 2, 1)
labels = ["Revenue growth\n(%)", "Failure rate\n(% pts)", "Maturity\nscore"]
coefs  = [ols_results[o]["coef"]  for o in OUTCOMES]
lo     = [ols_results[o]["ci_lo"] for o in OUTCOMES]
hi     = [ols_results[o]["ci_hi"] for o in OUTCOMES]
colors = ["#1D9E75" if c > 0 else "#D85A30" for c in coefs]
y_pos  = np.arange(len(OUTCOMES))
 
ax1.barh(y_pos, coefs, color=colors, alpha=0.85, height=0.5)
ax1.errorbar(
    coefs, y_pos,
    xerr=[[c - l for c, l in zip(coefs, lo)], [h - c for c, h in zip(coefs, hi)]],
    fmt="none", color="#333", capsize=4, linewidth=1.5,
)
ax1.axvline(0, color="gray", linewidth=0.8, linestyle="--")
ax1.set_yticks(y_pos)
ax1.set_yticklabels(labels, fontsize=10)
ax1.set_xlabel("Regression-adjusted effect of having ethics committee")
ax1.set_title("Effect of ethics committee\n(OLS + controls, 95% CI)")
 
x_max = max(hi) + abs(max(hi)) * 0.15
for o, pos in zip(OUTCOMES, y_pos):
    p   = ols_results[o]["pval"]
    sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
    ax1.text(x_max, pos, sig, va="center", fontsize=10, color="#333")

<Figure size 640x480 with 0 Axes>

In [19]:
# Plot 2: Revenue by maturity tier × ethics
ax2 = fig.add_subplot(2, 2, 2)
pivot.T.plot(kind="bar", ax=ax2, color=["#888780", "#378ADD"], alpha=0.85)
ax2.set_xlabel("AI maturity tier")
ax2.set_ylabel("Mean revenue growth (%)")
ax2.set_title("Revenue growth: maturity tier × ethics committee")
ax2.tick_params(axis="x", rotation=0)
ax2.legend(["No committee", "Has committee"], fontsize=9)

In [14]:
# Plot 3: Failure risk odds ratios
ax3 = fig.add_subplot(2, 2, 3)
risk_plot  = risk_fi.sort_values()
bar_colors = ["#1D9E75" if v < 1 else "#D85A30" for v in risk_plot]
ax3.barh(range(len(risk_plot)), risk_plot.values, color=bar_colors, alpha=0.85)
ax3.axvline(1, color="gray", linewidth=0.8, linestyle="--")
ax3.set_yticks(range(len(risk_plot)))
ax3.set_yticklabels(risk_plot.index, fontsize=9)
ax3.set_xlabel("Odds ratio  (>1 = increases failure risk)")
ax3.set_title("Failure risk drivers\n(logistic regression odds ratios)")

Text(0.5, 1.0, 'Failure risk drivers\n(logistic regression odds ratios)')

In [15]:
# Plot 4: Interaction line plot
ax4 = fig.add_subplot(2, 2, 4)
for has_eth, color, label in [
    (0, "#888780", "No committee"),
    (1, "#378ADD", "Has committee"),
]:
    x_vals, y_vals = interaction_data[has_eth]
    ax4.plot(x_vals, y_vals, "o-", color=color, label=label, linewidth=2, markersize=5)
ax4.set_xlabel("AI maturity score")
ax4.set_ylabel("Mean revenue growth (%)")
ax4.set_title("Revenue vs maturity × ethics (interaction)")
ax4.legend(fontsize=9)

In [20]:
plt.suptitle("Module 4 — Ethics & Risk Factor Analysis", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("outputs/m4_ethics_risk.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: outputs/m4_ethics_risk.png")

  Saved: outputs/m4_ethics_risk.png


In [21]:
with open("data/m4_results.pkl", "wb") as f:
    pickle.dump({
        "ols_results": ols_results,
        "risk_fi":     risk_fi,
        "pivot":       pivot,
    }, f)
print("  Saved: data/m4_results.pkl")

  Saved: data/m4_results.pkl
